## Assignment_6752300411 Lab 2 

## 🔧 DIY 1 — Trace Maximum Matching yourself

In [1]:
# ---- A tiny Thai dictionary (word list) -------------------------------------
# In practice you'd load ~60k words (e.g. PyThaiNLP's 'newmm' dictionary).
# We keep it small so every matching step is easy to follow by hand.
THAI_DICT = {
    "ตา",        # eye
    "ตาก",       # to air out / dry in the sun
    "กลม",       # round
    "ลม",        # wind
    "นอน",       # to lie down / sleep
    "หลับ",      # asleep
    "นอนหลับ",   # to sleep (compound)
    "อากาศ",     # weather / air
    "นั่ง",       # to sit
    "ริม",        # edge / bank
    "ทะเล",      # sea
}


def forward_max_match(text, dictionary, max_word_len=8, verbose=False):
    """Forward Maximum Matching (FMM): greedy longest match, left to right."""
    tokens, steps = [], []
    i = 0
    while i < len(text):
        matched = None
        for L in range(min(max_word_len, len(text) - i), 0, -1):
            candidate = text[i:i + L]
            if candidate in dictionary:
                matched = candidate
                steps.append((i, candidate, True))
                break
            steps.append((i, candidate, False))
        if matched is None:                 # unknown character: emit it alone
            matched = text[i]
            steps.append((i, matched, None))
        tokens.append(matched)
        i += len(matched)

    if verbose:
        print(f"FMM on {text!r}")
        for pos, cand, hit in steps:
            mark = "MATCH -> emit" if hit else ("no match -> emit single char" if hit is None else "not in dict")
            print(f"   i={pos:>2}  try {cand!r:<12} (len {len(cand)})  {mark}")
        print(f"   result: {'|'.join(tokens)}\n")
    return tokens


def backward_max_match(text, dictionary, max_word_len=8, verbose=False):
    """Backward Maximum Matching (BMM): greedy longest match, right to left."""
    tokens, steps = [], []
    j = len(text)
    while j > 0:
        matched = None
        for L in range(min(max_word_len, j), 0, -1):
            candidate = text[j - L:j]
            if candidate in dictionary:
                matched = candidate
                steps.append((j, candidate, True))
                break
            steps.append((j, candidate, False))
        if matched is None:
            matched = text[j - 1]
            steps.append((j, matched, None))
        tokens.insert(0, matched)
        j -= len(matched)

    if verbose:
        print(f"BMM on {text!r}")
        for pos, cand, hit in steps:
            mark = "MATCH -> emit" if hit else ("no match -> emit single char" if hit is None else "not in dict")
            print(f"   j={pos:>2}  try {cand!r:<12} (len {len(cand)})  {mark}")
        print(f"   result: {'|'.join(tokens)}\n")
    return tokens


print("Dictionary size:", len(THAI_DICT), "words")

Dictionary size: 11 words


In [2]:
# === DIY 1 ===
text = "อากาศตากลม"

## HINT
# fmm_out = ___(___, ___, verbose=True)
# bmm_out = ___(___, ___, verbose=True)
# they_disagree = (___ != ___)
##

# TODO 1: run forward_max_match on 'text' with THAI_DICT and verbose=True
fmm_out = forward_max_match(text, THAI_DICT, verbose=True)      # <-- replace None

# TODO 2: run backward_max_match on 'text' with THAI_DICT and verbose=True
bmm_out = backward_max_match(text, THAI_DICT, verbose=True)      # <-- replace None

# TODO 3: set this to True if the two algorithms produced DIFFERENT segmentations
they_disagree = (fmm_out != bmm_out)    # <-- replace None

print("FMM:", fmm_out)
print("BMM:", bmm_out)
print("Disagree?", they_disagree)

FMM on 'อากาศตากลม'
   i= 0  try 'อากาศตาก'   (len 8)  not in dict
   i= 0  try 'อากาศตา'    (len 7)  not in dict
   i= 0  try 'อากาศต'     (len 6)  not in dict
   i= 0  try 'อากาศ'      (len 5)  MATCH -> emit
   i= 5  try 'ตากลม'      (len 5)  not in dict
   i= 5  try 'ตากล'       (len 4)  not in dict
   i= 5  try 'ตาก'        (len 3)  MATCH -> emit
   i= 8  try 'ลม'         (len 2)  MATCH -> emit
   result: อากาศ|ตาก|ลม

BMM on 'อากาศตากลม'
   j=10  try 'กาศตากลม'   (len 8)  not in dict
   j=10  try 'าศตากลม'    (len 7)  not in dict
   j=10  try 'ศตากลม'     (len 6)  not in dict
   j=10  try 'ตากลม'      (len 5)  not in dict
   j=10  try 'ากลม'       (len 4)  not in dict
   j=10  try 'กลม'        (len 3)  MATCH -> emit
   j= 7  try 'อากาศตา'    (len 7)  not in dict
   j= 7  try 'ากาศตา'     (len 6)  not in dict
   j= 7  try 'กาศตา'      (len 5)  not in dict
   j= 7  try 'าศตา'       (len 4)  not in dict
   j= 7  try 'ศตา'        (len 3)  not in dict
   j= 7  try 'ตา'         (len 2) 

## 🔧 DIY 2 — Resolve an ambiguity with probabilities

In [3]:
import math

# ---- toy corpus counts (illustrative, N = 1,000,000 tokens) -----------------
N_TOKENS = 1000000 
UNIGRAM_COUNTS = {
    "ตา":     4200,   # eye        - common
    "กลม":    1800,   # round      - common
    "ตาก":     320,   # to air out - rarer
    "ลม":     3600,   # wind       - common
    "นั่ง":    2000,   # to sit
    "นอน":    5100,
    "หลับ":   1400,
    "นอนหลับ": 900,
    "อากาศ":  2900,
}

def unigram_prob(word):
    return UNIGRAM_COUNTS.get(word, 1) / N_TOKENS      # crude add-one-ish floor

def score_segmentation(tokens):
    """Return (probability, log10 probability) of a segmentation under the unigram LM."""
    p = 1.0
    for w in tokens:
        p *= unigram_prob(w)
    return p, math.log10(p)

# 2 candidates to find prob
cand_A = ["ตา", "กลม"]     # BMM's answer: 'round eyes'
cand_B = ["ตาก", "ลม"]     # FMM's answer: 'airing in the wind'

for name, cand in [("A (ตา|กลม)", cand_A), ("B (ตาก|ลม)", cand_B)]:
    parts = "  x  ".join(f"P({w})={unigram_prob(w):.5f}" for w in cand)
    p, logp = score_segmentation(cand)
    print(f"{name:<14} {parts}   =  {p:.4e}   (log10 = {logp:.4f})")

pA, _ = score_segmentation(cand_A)
pB, _ = score_segmentation(cand_B)
winner = cand_A if pA > pB else cand_B
print(f"\nWinner (unigram): {'|'.join(winner)}   by a factor of {max(pA, pB) / min(pA, pB):.4f}x")

A (ตา|กลม)     P(ตา)=0.00420  x  P(กลม)=0.00180   =  7.5600e-06   (log10 = -5.1215)
B (ตาก|ลม)     P(ตาก)=0.00032  x  P(ลม)=0.00360   =  1.1520e-06   (log10 = -5.9385)

Winner (unigram): ตา|กลม   by a factor of 6.5625x


In [4]:
# === DIY 2 ===
UNIGRAM_COUNTS["ลมกลม"] = 50      # register the hypothetical compound

one_word = ["ลมกลม"]
two_words = ["ลม", "กลม"]

## HINT
# p_one, log_one = ___(one_word)
# p_two, log_two = score_segmentation(___)

# TODO 1: score both analyses with score_segmentation()
p_one, log_one = score_segmentation(one_word)
p_two, log_two = score_segmentation(two_words)

##HINT
# winner = one_word if ____ > _____ else two_words
# factor = ___(p_one, p_two) / ___(p_one, p_two)

# TODO 2: which analysis wins, and by what factor?
winner = one_word if p_one > p_two else two_words    # <-- replace with one_word or two_words
factor = max(p_one, p_two) / min(p_one, p_two)       # <-- replace with the ratio (>1)

print(f"one word : {p_one}")
print(f"two words: {p_two}")
print(f"winner   : {winner}  by {factor}x")

one word : 5e-05
two words: 6.48e-06
winner   : ['ลมกลม']  by 7.71604938271605x


## 🔧 DIY 3 — Compute a Viterbi cell by hand

In [5]:
# ---- The HMM: J&M Ch.18 Fig 18.12 (A) and Fig 18.13 (B), verbatim ---------
TAGS = ["NNP", "MD", "VB", "JJ", "NN", "RB", "DT"]

A = {
    "<s>": {"NNP": 0.2767, "MD": 0.0006, "VB": 0.0031, "JJ": 0.0453, "NN": 0.0449, "RB": 0.0510, "DT": 0.2026},
    "NNP": {"NNP": 0.3777, "MD": 0.0110, "VB": 0.0009, "JJ": 0.0084, "NN": 0.0584, "RB": 0.0090, "DT": 0.0025},
    "MD":  {"NNP": 0.0008, "MD": 0.0002, "VB": 0.7968, "JJ": 0.0005, "NN": 0.0008, "RB": 0.1698, "DT": 0.0041},
    "VB":  {"NNP": 0.0322, "MD": 0.0005, "VB": 0.0050, "JJ": 0.0837, "NN": 0.0615, "RB": 0.0514, "DT": 0.2231},
    "JJ":  {"NNP": 0.0366, "MD": 0.0004, "VB": 0.0001, "JJ": 0.0733, "NN": 0.4509, "RB": 0.0036, "DT": 0.0036},
    "NN":  {"NNP": 0.0096, "MD": 0.0176, "VB": 0.0014, "JJ": 0.0086, "NN": 0.1216, "RB": 0.0177, "DT": 0.0068},
    "RB":  {"NNP": 0.0068, "MD": 0.0102, "VB": 0.1011, "JJ": 0.1012, "NN": 0.0120, "RB": 0.0728, "DT": 0.0479},
    "DT":  {"NNP": 0.1147, "MD": 0.0021, "VB": 0.0002, "JJ": 0.2157, "NN": 0.4744, "RB": 0.0102, "DT": 0.0017},
}

B = {
    "NNP": {"Janet": 0.000032, "will": 0.0,      "back": 0.0,      "the": 0.000048, "bill": 0.0},
    "MD":  {"Janet": 0.0,      "will": 0.308431, "back": 0.0,      "the": 0.0,      "bill": 0.0},
    "VB":  {"Janet": 0.0,      "will": 0.000028, "back": 0.000672, "the": 0.0,      "bill": 0.000028},
    "JJ":  {"Janet": 0.0,      "will": 0.0,      "back": 0.000340, "the": 0.0,      "bill": 0.0},
    "NN":  {"Janet": 0.0,      "will": 0.000200, "back": 0.000223, "the": 0.0,      "bill": 0.002337},
    "RB":  {"Janet": 0.0,      "will": 0.0,      "back": 0.010446, "the": 0.0,      "bill": 0.0},
    "DT":  {"Janet": 0.0,      "will": 0.0,      "back": 0.0,      "the": 0.506099, "bill": 0.0},
}

SENTENCE = ["Janet", "will", "back", "the", "bill"]
GOLD_TAGS = ["NNP", "MD", "VB", "DT", "NN"]      # J&M eq. 18.20

# quick structural checks
assert len(TAGS) == 7 and len(SENTENCE) == 5
assert A["MD"]["VB"] == 0.7968 and B["DT"]["the"] == 0.506099
print(f"HMM ready: N = {len(TAGS)} tags, T = {len(SENTENCE)} words")
print(f"P(VB|MD)   = {A['MD']['VB']}      <- after a modal, expect a base-form verb")
print(f"P(NN|DT)   = {A['DT']['NN']}      <- after a determiner, expect a noun")
print(f"P(the|DT)  = {B['DT']['the']}")

HMM ready: N = 7 tags, T = 5 words
P(VB|MD)   = 0.7968      <- after a modal, expect a base-form verb
P(NN|DT)   = 0.4744      <- after a determiner, expect a noun
P(the|DT)  = 0.506099


In [6]:
# === DIY 3 ===
v1_nnp = A["<s>"]["NNP"] * B["NNP"]["Janet"]     # given: 8.8544e-6


## HINT
# a_nnp_nn = A["___"]["___"]       # 0.0584 
# b_nn_will = B["___"]["___"]      # 0.000200 
# TODO 1: look up the two probabilities you need
a_nnp_nn = A["NNP"]["NN"]      # <-- P(NN | NNP) from Figure 18.12
b_nn_will = B["NN"]["will"]     # <-- P(will | NN) from Figure 18.13


## HINT
# v2_nn = v1_nnp * a_nnp_nn * b_nn_will
# v2_md = v1_nnp * A["NNP"]["MD"] * B["MD"]["will"]
# ratio = v2_md / v2_nn

# TODO 2: compute v2(NN)
v2_nn = v1_nnp * a_nnp_nn * b_nn_will         # <-- replace

# TODO 3: how many times larger is v2(MD) than v2(NN)?
v2_md = v1_nnp * A["NNP"]["MD"] * B["MD"]["will"]
ratio = v2_md / v2_nn         # <-- replace

print(f"v2(NN) = {v2_nn}")
print(f"v2(MD) = {v2_md:.6e}")
print(f"MD is {ratio} times more likely than NN")

v2(NN) = 1.03419392e-10
v2(MD) = 3.004069e-08
MD is 290.4744006849315 times more likely than NN


## 🔧 DIY 4 — BIO-tag a Thai sentence

In [7]:
# ---- BIO utilities --------------------------------------------------------
def spans_to_bio(tokens, spans):
    """spans: list of (start, end_inclusive, type) -> BIO tag list."""
    tags = ["O"] * len(tokens)
    for start, end, etype in spans:
        tags[start] = f"B-{etype}"
        for i in range(start + 1, end + 1):
            tags[i] = f"I-{etype}"
    return tags


def bio_to_spans(tags):
    """BIO tag list -> list of (start, end_inclusive, type). Inverse of the above."""
    spans, start, etype = [], None, None
    for i, tag in enumerate(tags):
        if tag.startswith("B-"):
            if start is not None:
                spans.append((start, i - 1, etype))
            start, etype = i, tag[2:]
        elif tag.startswith("I-") and start is not None and tag[2:] == etype:
            continue                              # still inside the same entity
        else:                                     # O, or an inconsistent I-
            if start is not None:
                spans.append((start, i - 1, etype))
            start, etype = None, None
    if start is not None:
        spans.append((start, len(tags) - 1, etype))
    return spans


def bio_to_io(tags):
    return [t.replace("B-", "I-") for t in tags]


def bio_to_bioes(tags):
    """B-X I-X I-X -> B-X I-X E-X ;  a lone B-X -> S-X"""
    out = []
    spans = bio_to_spans(tags)
    out = ["O"] * len(tags)
    for start, end, etype in spans:
        if start == end:
            out[start] = f"S-{etype}"
        else:
            out[start] = f"B-{etype}"
            for i in range(start + 1, end):
                out[i] = f"I-{etype}"
            out[end] = f"E-{etype}"
    return out


# Reproduce J&M Figure 18.7
tokens = ["Jane", "Villanueva", "of", "United", "Airlines", "Holding",
          "discussed", "the", "Chicago", "route", "."]
spans = [(0, 1, "PER"), (3, 5, "ORG"), (8, 8, "LOC")]

bio = spans_to_bio(tokens, spans)
io = bio_to_io(bio)
bioes = bio_to_bioes(bio)

print(f"{'Words':<12}{'IO Label':<12}{'BIO Label':<12}{'BIOES Label':<12}")
print("-" * 48)
for tok, a, b, c in zip(tokens, io, bio, bioes):
    print(f"{tok:<12}{a:<12}{b:<12}{c:<12}")

# round-trip check: BIO -> spans -> BIO
assert bio_to_spans(bio) == spans
print(f"\n[ok] BIO round-trips to the original spans: {bio_to_spans(bio)}")

n_types = 3      # PER, ORG, LOC
print(f"\nTag inventory for {n_types} entity types: 2n+1 = {2 * n_types + 1} tags")
print(f"  {sorted(set(f'{p}-{t}' for p in 'BI' for t in ['PER','ORG','LOC']) | {'O'})}")
print("\nWhy IO is weaker: it cannot separate two adjacent same-type entities.")
adjacent = ["Bangkok", "Chiang", "Mai"]                     # two LOCs side by side
adj_bio = spans_to_bio(adjacent, [(0, 0, "LOC"), (1, 2, "LOC")])
print(f"  BIO {adj_bio} -> spans {bio_to_spans(adj_bio)}  (2 entities, correct)")
print(f"  IO  {bio_to_io(adj_bio)} -> indistinguishable from ONE 3-token LOC")

Words       IO Label    BIO Label   BIOES Label 
------------------------------------------------
Jane        I-PER       B-PER       B-PER       
Villanueva  I-PER       I-PER       E-PER       
of          O           O           O           
United      I-ORG       B-ORG       B-ORG       
Airlines    I-ORG       I-ORG       I-ORG       
Holding     I-ORG       I-ORG       E-ORG       
discussed   O           O           O           
the         O           O           O           
Chicago     I-LOC       B-LOC       S-LOC       
route       O           O           O           
.           O           O           O           

[ok] BIO round-trips to the original spans: [(0, 1, 'PER'), (3, 5, 'ORG'), (8, 8, 'LOC')]

Tag inventory for 3 entity types: 2n+1 = 7 tags
  ['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']

Why IO is weaker: it cannot separate two adjacent same-type entities.
  BIO ['B-LOC', 'B-LOC', 'I-LOC'] -> spans [(0, 0, 'LOC'), (1, 2, 'LOC')]  (2 entities, co

In [8]:
# === DIY 6 ===
th_tokens = ["บริษัท", "ปูนซิเมนต์", "ไทย", "ตั้งอยู่", "ที่", "กรุงเทพมหานคร"]
#   index:       0          1          2       3        4          5
# gold entities: (1,2,'ORG')  and  (5,5,'LOC')

##HINT
# th_bio = ["O", "B-ORG", "I-ORG", "O", "O", "B-LOC"]
# th_bioes = bio_to_bioes(_______)
# recovered = bio_to_spans(______)

# TODO 1: write the BIO tags by hand (6 strings)
th_bio = ["O", "B-ORG", "I-ORG", "O", "O", "B-LOC"]      # <-- e.g. ["O", "B-ORG", ...]

# TODO 2: convert to BIOES using bio_to_bioes()
th_bioes = bio_to_bioes(th_bio)

# TODO 3: recover the spans and check they match the gold entities
recovered = bio_to_spans(th_bioes)

print("BIO  :", th_bio)
print("BIOES:", th_bioes)
print("spans:", recovered)

BIO  : ['O', 'B-ORG', 'I-ORG', 'O', 'O', 'B-LOC']
BIOES: ['O', 'B-ORG', 'E-ORG', 'O', 'O', 'S-LOC']
spans: [(1, 1, 'ORG')]


## 🔧 DIY 5 — Compute Precision, Recall, and F1

In [9]:
# === DIY 5 ===
gold_set = {("สมชาย", "PER"), ("เชียงใหม่", "LOC"), ("ปตท.", "ORG"),
            ("กรุงเทพ", "LOC"), ("2026", "DATE")}
pred_set = {("สมชาย", "PER"), ("เชียง", "LOC"), ("ปตท.", "ORG"),
            ("กรุงเทพ", "ORG")}

##HINT 
# tp_set = __ & __
# fp_set = __ - __
# fn_set = __ - __
# tp, fp, fn = len(tp_set), len(fp_set), len(fn_set)

# precision = __ / (__ + __)
# recall = __ / (__ + __)
# f1 = 2 * __ * __ / (__ + __)

# TODO 1: compute the three counts
tp = len(gold_set & pred_set)
fp = len(pred_set - gold_set)
fn = len(gold_set - pred_set)

# TODO 2: compute the three metrics
precision = tp /(tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

print(f"TP={tp} FP={fp} FN={fn}")
print(f"P={precision} R={recall} F1={f1}")

TP=2 FP=2 FN=3
P=0.5 R=0.4 F1=0.4444444444444445
